<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 9A · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">保留输入、分流拒收、自动验收</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:900px;margin:0">使用统一订单样本，观察 SQL、结果与验收证据。请按顺序运行单元。</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">Doris 4.1.3 target · Order data · Isolated course database</span>
</div>

By the end of this lab, you will have a running Doris environment, an `events` table containing more than 10 million rows, and analytical results produced from that table. Run the cells in order.

[讲义](course9a_data_quality_and_schema_validation.md) · [课程入口](../README.md)


## 实验范围

重建 d09_raw、d09_classified 视图、orders_clean_demo 和 orders_reject_demo。12 行输入包括两种不同的异常，先全部暂存，再做业务准入。D06 读取本 Lab 的合格输出。


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course import WarehouseLab
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows
from dw_course.ui import show_sql, show_response

lab = WarehouseLab()




## 1. 原始字段先保留

暂存金额和订单号为字符串。D05 的整批失败演示与本节显式分流不是同一机制。


In [ ]:
lab.execute("DROP VIEW IF EXISTS d09_classified")
lab.execute("DROP TABLE IF EXISTS d09_raw")
fields = ", ".join(col + " VARCHAR(100) NULL" for col in ORDER_COLUMNS)
lab.execute(f'CREATE TABLE d09_raw (input_id BIGINT NOT NULL, {fields}) DUPLICATE KEY(input_id) DISTRIBUTED BY HASH(input_id) BUCKETS 1 PROPERTIES("replication_num"="1")')
raw = fixture("raw_orders.json")
lab.insert("d09_raw", ["input_id", *ORDER_COLUMNS],
           [(r["input_id"], *[None if r[col] is None else str(r[col]) for col in ORDER_COLUMNS]) for r in raw])
expect(lab.query("SELECT COUNT(*) FROM d09_raw"), [(12,)])


## 2. 按固定规则分流

TRY_CAST 将本样本的非法格式变为 NULL，以便保留原因。生产准入还需覆盖日期、客户、版本等字段；本节只针对已声明的两种脏数据。


In [ ]:
lab.execute("""
CREATE VIEW d09_classified AS
SELECT *, CASE
    WHEN TRY_CAST(order_id AS BIGINT) IS NULL THEN 'INVALID_ORDER_ID'
    WHEN TRY_CAST(order_amount AS DECIMAL(12,2)) IS NULL
      OR TRY_CAST(order_amount AS DECIMAL(12,2)) < 0 THEN 'INVALID_AMOUNT'
    ELSE NULL END AS reject_reason
FROM d09_raw
""")
lab.execute("DROP TABLE IF EXISTS orders_clean_demo")
ddl = order_ddl("orders_clean_demo")
show_sql("建表 SQL", ddl)
lab.execute(ddl)
lab.execute("DROP TABLE IF EXISTS orders_reject_demo")
lab.execute('CREATE TABLE orders_reject_demo (input_id BIGINT, reason VARCHAR(32)) DUPLICATE KEY(input_id) DISTRIBUTED BY HASH(input_id) BUCKETS 1 PROPERTIES("replication_num"="1")')
lab.execute(f"INSERT INTO orders_clean_demo ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM d09_classified WHERE reject_reason IS NULL")
lab.execute("INSERT INTO orders_reject_demo SELECT input_id, reject_reason FROM d09_classified WHERE reject_reason IS NOT NULL")


## 3. 检查业务质量并注入错误

数量只是第一层校验，还要比对完整记录。事件时间检查用固定截止时刻，不冒充真实端到端接入延迟。


In [ ]:
expected_rows = order_rows(fixture("orders.json"))
def quality_report():
    expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_clean_demo"), [(10,"1400.00")])
    expect(lab.query("SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM orders_clean_demo"), [(0,)])
    expect(lab.query("SELECT COUNT(*) FROM orders_clean_demo WHERE event_time > '2026-01-02 12:00:00' OR event_time IS NULL"), [(0,)])
    expect(lab.query("SELECT COUNT(*) FROM orders_clean_demo WHERE status <> 'CREATED' OR paid_amount <> 0 OR refund_amount <> 0"), [(0,)])
    # DATE_FORMAT makes the transport representation explicit for fixture comparison.
    projection = ",".join("DATE_FORMAT(event_time, '%Y-%m-%d %H:%i:%s')" if col == "event_time" else col for col in ORDER_COLUMNS)
    expect(lab.query(f"SELECT {projection} FROM orders_clean_demo ORDER BY order_id"), expected_rows)

expect(lab.query("SELECT input_id, reason FROM orders_reject_demo ORDER BY input_id"),
       [(11,"INVALID_AMOUNT"),(12,"INVALID_ORDER_ID")])
expect(lab.query("SELECT COUNT(*) FROM d09_classified WHERE reject_reason IS NULL"), [(10,)])
expect(lab.query("SELECT COUNT(*) FROM d09_raw r LEFT JOIN orders_reject_demo x ON r.input_id=x.input_id WHERE x.input_id IS NULL"), [(10,)])
quality_report()

# Inject one duplicate: the validator must detect it, not just print a warning.
lab.insert("orders_clean_demo", ORDER_COLUMNS, [expected_rows[0]])
try:
    quality_report()
except AssertionError as error:
    print("Expected quality failure:", error)
else:
    raise AssertionError("Injected duplicate was not detected")
lab.execute("TRUNCATE TABLE orders_clean_demo")
lab.execute(f"INSERT INTO orders_clean_demo ({','.join(ORDER_COLUMNS)}) SELECT {','.join(ORDER_COLUMNS)} FROM d09_classified WHERE reject_reason IS NULL")
quality_report()
lab.close()


## 完成与排查

12 行输入、10 行合格、2 行拒收，合格记录逐行匹配；拒收原因可通过 input_id 回查原始字段。先保留再分流才可追踪。新增其他脏数据类型时要同时新增规则与独立预期结果。
